# Fig 5a - cohort & design (transparent about confounds)

Curated <=06Jul cohort (n=71). Shows Diagnosis, and the confounds openly: Sex x Diagnosis, the non-overlapping Age distribution, and Batch x Diagnosis. Kernel: downstr.


## Locations

Paths are relative to the data directory given by the environment variable `M26_DATA_ROOT` (see the README); figure panels are written to `M26_RESULTS_ROOT`.

In [ ]:
# Locations (see README): M26_DATA_ROOT is required; M26_RESULTS_ROOT defaults to <repo>/results
import os
from pathlib import Path

REPO_ROOT = os.environ.get("M26_REPO_ROOT") or str(
    next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "environment" / "downstr.yml").is_file())
)
os.environ["M26_REPO_ROOT"] = REPO_ROOT
DATA_ROOT = os.environ["M26_DATA_ROOT"]
RESULTS_ROOT = os.environ.get("M26_RESULTS_ROOT", f"{REPO_ROOT}/results")

In [ ]:
import sys, warnings

warnings.filterwarnings("ignore")
import anndata as ad, numpy as np, pandas as pd, matplotlib.pyplot as plt

sys.path.insert(0, f"{REPO_ROOT}/figures/style")
import paper_fig_style as S

S.apply_rcparams()
PANEL = "fig5a_cohort_design"
ASD, CTRL = "#d73027", "#4575b4"
a = ad.read_h5ad(
    f"{DATA_ROOT}/3utr/Blood/all_till_20240724/utrcalling/UTR_count_table_genelevel_bw12_above10cts_apa_named.h5ad",
    backed="r",
)
o = a.obs[a.obs.status_split.isin(["Train", "Test_original"])].copy()
o["Diagnosis"] = o.Diagnosis.astype(str)
o["age"] = pd.to_numeric(o.age_specific, errors="coerce")
print("n=", len(o))

In [ ]:
test = a.obs[a.obs.status_split.isin(["Train", "Test_original"])].copy()
test.head()

In [ ]:
test.age_specific.tolist()

In [ ]:
o.Diagnosis.value_counts(dropna=True).sum()

In [ ]:
fig, ax = plt.subplots(1, 4, figsize=(S.FULL_W_PX / 300, S.mm_to_px(45) / 300))
cols = {"ASD": ASD, "CTRL": CTRL}
# (1) diagnosis counts
vc = o.Diagnosis.value_counts()
ax[0].bar(vc.index, vc.values, color=[cols[k] for k in vc.index])
ax[0].set_title("Diagnosis", fontsize=S.FONT_LABEL)
ax[0].set_ylabel("n samples", fontsize=S.FONT_LABEL)
for i, (k, v) in enumerate(vc.items()):
    ax[0].text(i, v + 0.5, str(v), ha="center", fontsize=S.FONT_TICK)
# (2) Sex x Diagnosis
ct = pd.crosstab(o.Diagnosis, o.Sex)
btm = np.zeros(len(ct))
for sx, c in zip(["Female", "Male"], ["#f1a340", "#998ec3"]):
    if sx in ct:
        ax[1].bar(ct.index, ct[sx], bottom=btm, label=sx, color=c)
        btm = btm + ct[sx].values
ax[1].set_title("Sex x Diagnosis", fontsize=S.FONT_LABEL)
ax[1].legend(fontsize=S.FONT_LEGEND, frameon=False)
# (3) Age by Diagnosis
for k in ["CTRL", "ASD"]:
    y = o.loc[o.Diagnosis == k, "age"].dropna()
    x = np.random.normal({"CTRL": 0, "ASD": 1}[k], 0.07, len(y))
    ax[2].scatter(x, y, s=10, c=cols[k], alpha=0.7, edgecolors="none")
ax[2].set_xticks([0, 1])
ax[2].set_xticklabels(["CTRL", "ASD"])
ax[2].set_ylabel("age (yr)", fontsize=S.FONT_LABEL)
ax[2].set_title("Age (yr)", fontsize=S.FONT_LABEL)
# (4) Batch x Diagnosis
cb = pd.crosstab(o.Batch, o.Diagnosis)
cb = cb[cb.sum(1) >= 2]
x = np.arange(len(cb))
ax[3].bar(x, cb.get("CTRL", 0), color=CTRL, label="CTRL")
ax[3].bar(x, cb.get("ASD", 0), bottom=cb.get("CTRL", 0), color=ASD, label="ASD")
ax[3].set_xticks(x)
ax[3].set_xticklabels(
    [b.split(":")[0].replace("VH00", "") + ":" + b.split(":")[1] for b in cb.index],
    rotation=45,
    fontsize=S.FONT_TICK - 1,
)
ax[3].set_title("Batch x Diagnosis", fontsize=S.FONT_LABEL)
ax[3].legend(fontsize=S.FONT_LEGEND, frameon=False)
for x_ in ax:
    x_.tick_params(labelsize=S.FONT_TICK)
    [s.set_visible(False) for s in (x_.spines["top"], x_.spines["right"])]
fig.subplots_adjust(left=0.06, right=0.98, bottom=0.18, top=0.88, wspace=0.45)
S.save_and_show(fig, PANEL, figure="Fig5")

### Size variants: quarter-height full-width (1×4) and quarter-height half-width (2×2)


In [ ]:
# --- Size variants of fig5a: quarter-height full-width (1x4) + quarter-height half-width (2x2) ---
def _fig5a_row(figsize, save_name):
    cols = {"ASD": ASD, "CTRL": CTRL}
    fig, ax = plt.subplots(1, 4, figsize=figsize)
    vc = o.Diagnosis.value_counts()
    ax[0].bar(vc.index, vc.values, color=[cols[k] for k in vc.index])
    ax[0].set_title("Diagnosis", fontsize=S.FONT_LABEL)
    ax[0].set_ylabel("n samples", fontsize=S.FONT_LABEL)
    for i, (k, v) in enumerate(vc.items()):
        ax[0].text(i, v + 0.5, str(v), ha="center", fontsize=S.FONT_TICK)
    ct = pd.crosstab(o.Diagnosis, o.Sex)
    btm = np.zeros(len(ct))
    for sx, c in zip(["Female", "Male"], ["#f1a340", "#998ec3"]):
        if sx in ct:
            ax[1].bar(ct.index, ct[sx], bottom=btm, label=sx, color=c)
            btm = btm + ct[sx].values
    ax[1].set_title("Sex × Diagnosis", fontsize=S.FONT_LABEL)
    ax[1].legend(fontsize=S.FONT_LEGEND, frameon=False)
    for k in ["CTRL", "ASD"]:
        y = o.loc[o.Diagnosis == k, "age"].dropna()
        x = np.random.normal({"CTRL": 0, "ASD": 1}[k], 0.07, len(y))
        ax[2].scatter(x, y, s=10, c=cols[k], alpha=0.7, edgecolors="none")
    ax[2].set_xticks([0, 1])
    ax[2].set_xticklabels(["CTRL", "ASD"])
    ax[2].set_ylabel("age (yr)", fontsize=S.FONT_LABEL)
    ax[2].set_title("Age (yr)", fontsize=S.FONT_LABEL)
    cb = pd.crosstab(o.Batch, o.Diagnosis)
    cb = cb[cb.sum(1) >= 2]
    x = np.arange(len(cb))
    ax[3].bar(x, cb.get("CTRL", 0), color=CTRL, label="CTRL")
    ax[3].bar(x, cb.get("ASD", 0), bottom=cb.get("CTRL", 0), color=ASD, label="ASD")
    ax[3].set_xticks(x)
    ax[3].set_xticklabels(
        [b.split(":")[0].replace("VH00", "") + ":" + b.split(":")[1] for b in cb.index],
        rotation=45,
        ha="right",
        fontsize=S.FONT_TICK - 1,
    )
    ax[3].set_title("Batch × Diagnosis", fontsize=S.FONT_LABEL)
    ax[3].legend(fontsize=S.FONT_LEGEND, frameon=False)
    for x_ in ax:
        x_.tick_params(labelsize=S.FONT_TICK)
        [s.set_visible(False) for s in (x_.spines["top"], x_.spines["right"])]
    fig.subplots_adjust(left=0.06, right=0.98, bottom=0.18, top=0.88, wspace=0.45)
    S.save_and_show(fig, save_name, figure="Fig5")


def _fig5a_sq(figsize, save_name):
    cols = {"ASD": ASD, "CTRL": CTRL}
    fig, ax = plt.subplots(2, 2, figsize=figsize)
    ax = ax.ravel()
    vc = o.Diagnosis.value_counts()
    ax[0].bar(vc.index, vc.values, color=[cols[k] for k in vc.index])
    ax[0].set_ylim(0, vc.max() * 1.18)
    ax[0].set_title("Diagnosis", fontsize=S.FONT_LABEL)
    ax[0].set_ylabel("n samples", fontsize=S.FONT_LABEL)
    for i, (k, v) in enumerate(vc.items()):
        ax[0].text(i, v + 0.5, str(v), ha="center", fontsize=S.FONT_TICK)
    ct = pd.crosstab(o.Diagnosis, o.Sex)
    btm = np.zeros(len(ct))
    for sx, c in zip(["Female", "Male"], ["#f1a340", "#998ec3"]):
        if sx in ct:
            ax[1].bar(ct.index, ct[sx], bottom=btm, label=sx, color=c)
            btm = btm + ct[sx].values
    ax[1].set_ylim(0, btm.max() * 1.28)
    ax[1].set_title("Sex × Diagnosis", fontsize=S.FONT_LABEL)
    ax[1].legend(
        fontsize=S.FONT_LEGEND - 0.5,
        frameon=False,
        loc="upper right",
        handlelength=0.9,
        handletextpad=0.4,
        labelspacing=0.25,
        borderpad=0.1,
    )
    for k in ["CTRL", "ASD"]:
        y = o.loc[o.Diagnosis == k, "age"].dropna()
        x = np.random.normal({"CTRL": 0, "ASD": 1}[k], 0.07, len(y))
        ax[2].scatter(x, y, s=8, c=cols[k], alpha=0.7, edgecolors="none")
    ax[2].set_xticks([0, 1])
    ax[2].set_xticklabels(["CTRL", "ASD"])
    ax[2].set_ylabel("age (yr)", fontsize=S.FONT_LABEL)
    ax[2].set_title("Age (yr)", fontsize=S.FONT_LABEL)
    cb = pd.crosstab(o.Batch, o.Diagnosis)
    cb = cb[cb.sum(1) >= 2]
    x = np.arange(len(cb))
    ax[3].bar(x, cb.get("CTRL", 0), color=CTRL)
    ax[3].bar(x, cb.get("ASD", 0), bottom=cb.get("CTRL", 0), color=ASD)
    ax[3].set_ylim(0, cb.sum(1).max() * 1.12)
    ax[3].set_xticks(x)
    ax[3].set_xticklabels(
        [b.split(":")[0].replace("VH00", "") + ":" + b.split(":")[1] for b in cb.index],
        rotation=45,
        ha="right",
        fontsize=S.FONT_TICK - 1,
    )
    ax[3].set_title(
        "Batch × Diagnosis", fontsize=S.FONT_LABEL
    )  # CTRL=blue / ASD=red already keyed in Diagnosis panel
    for x_ in ax:
        x_.tick_params(labelsize=S.FONT_TICK)
        [s.set_visible(False) for s in (x_.spines["top"], x_.spines["right"])]
    fig.subplots_adjust(
        left=0.10, right=0.97, bottom=0.15, top=0.91, wspace=0.5, hspace=0.7
    )
    S.save_and_show(fig, save_name, figure="Fig5")


_fig5a_row(
    (S.FULL_W_PX / 300, S.QUARTER_H_PX / 300), "fig5a_cohort_design__quarter_full"
)
_fig5a_sq(
    (S.HALF_W_PX / 300, S.QUARTER_H_PX / 300), "fig5a_cohort_design__quarter_half"
)